## Importação de Bibliotecas

In [ ]:
# Dependências
import sys
#!{sys.executable} -m pip install --disable-pip-version-check -r ../requirements.txt -q
print('Bibliotecas instaladas')

In [ ]:
# Acesso aos módulos do diretório
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
import os

# Manipulação dos dados 
import pandas as pd
import numpy as np
from datetime import datetime
import pickle

# Visualização dos dados
import matplotlib.pyplot as plt
import seaborn as sns

# Funções customizadas
from configs.paths import *
from configs.function_basic import *
from configs.function_others import *

# Modelos
import statsmodels.api as sm

# Métricas e validação
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report, ConfusionMatrixDisplay

# Avisos
import warnings
warnings.filterwarnings('ignore')

# Configuração
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print('✅ Bibliotecas carregadas com sucesso')


### Parâmetros globais

In [ ]:
# define a coluna alvo do modelo
TARGET = 'FPD'
cutoff = 0.4
DATA_EXECUCAO = datetime.now().strftime('%d-%m-%Y')
VERSAO = 'V1 - baseline'

## Carregamento dos dados

In [ ]:
# Carregar dados
abt01_train = pd.read_parquet(PROCESSED_DIR / 'abt01_train_tr.parquet')
abt01_test = pd.read_parquet(PROCESSED_DIR / 'abt01_test_tr.parquet')

# Distribuição target
target_dist = abt01_train[TARGET].value_counts()
target_pct = (abt01_train[TARGET].value_counts(normalize=True) * 100).round(2)

print(f'\n📊 Distribuição do Target (FPD):')
print(f"   Não-inadimplentes (0): {target_dist[0]:,} ({target_pct[0]:.2f}%)")
print(f"   Inadimplentes (1):     {target_dist[1]:,} ({target_pct[1]:.2f}%)")
print(f"   Taxa de Inadimplência: {target_pct[1]:.2f}%")


## REGRESSÃO LOGÍSTICA

In [ ]:
# carrega a lista de features usadas pelo modelo
with open(Path(ARTIFACT_DIR) / 'model_features.pkl', 'rb') as f:
    lista_features_rl = pickle.load(f)

print(lista_features_rl)

In [ ]:
# ajusta regressão logística com scorecard estatístico (Wald, p-value)

def logistic_regression_with_scorecard(data, target_var, features):
    # separa variáveis explicativas (X) e target (y)
    X = data[features].copy()
    y = data[target_var]

    # adiciona intercepto ao modelo
    X = sm.add_constant(X)

    # remove linhas com NaN ou infinito para garantir convergência do modelo
    mask = np.isfinite(X).all(axis=1) & np.isfinite(y)
    X = X[mask]
    y = y[mask]

    # ajusta o modelo de regressão logística
    model = sm.Logit(y, X).fit(disp=False)

    # extrai tabela estatística do modelo
    summary = model.summary2().tables[1]

    # calcula estatística de Wald (z²)
    summary['Wald'] = summary['z'] ** 2

    # monta scorecard com métricas relevantes e remove intercepto
    scorecard = (
        summary
        .loc[summary.index != 'const', ['Coef.', 'P>|z|', 'Wald']]
        .rename(columns={
            'Coef.': 'Beta Coefficient',
            'P>|z|': 'P-Value',
            'Wald': 'Wald Statistic'
        })
        .sort_values(by='Wald Statistic', ascending=False)
    )

    # retorna modelo treinado e scorecard ordenado por poder estatístico
    return model, scorecard

In [ ]:
# treina regressão logística final e gera scorecard do modelo
model, scorecard = logistic_regression_with_scorecard(abt01_train, TARGET, lista_features_rl)
scorecard

In [ ]:
# gera probabilidades previstas (PD) para treino e teste
X_train = abt01_train[lista_features_rl]
X_test  = abt01_test[lista_features_rl]

X_train = sm.add_constant(X_train)
X_test  = sm.add_constant(X_test)

abt01_train['PD'] = model.predict(X_train)
abt01_test['PD']  = model.predict(X_test)

abt01_train[['PD']].head()

In [ ]:
# converte PD em log-odds para construção do scorecard
abt01_train['LOG_ODDS'] = np.log(abt01_train['PD'] / (1 - abt01_train['PD']))
abt01_test['LOG_ODDS']  = np.log(abt01_test['PD'] / (1 - abt01_test['PD']))

abt01_train[['PD', 'LOG_ODDS']].head()

In [ ]:
# converte log-odds em score de crédito na escala definida
PDO = 50
BASE_SCORE = 600
BASE_ODDS = 50  # 50:1 (bons : maus)

factor = PDO / np.log(2)
offset = BASE_SCORE - factor * np.log(BASE_ODDS)

abt01_train['SCORE'] = offset - factor * abt01_train['LOG_ODDS']
abt01_test['SCORE']  = offset - factor * abt01_test['LOG_ODDS']

abt01_train[['PD', 'LOG_ODDS', 'SCORE']].head()


In [ ]:
# valida ordenação do score em relação ao risco
abt01_train[['PD', 'SCORE']].corr()

## Métricas

In [ ]:
# encontra cutoff que maximiza KS
def ks_by_cutoff(y_true, pd_score, cutoffs):
    ks_results = []

    for c in cutoffs:
        pred = (pd_score >= c).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()

        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0  # recall maus
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0  # erro bons

        ks = abs(tpr - fpr)

        ks_results.append({'cutoff': c, 'KS': ks})

    return pd.DataFrame(ks_results)

cutoffs = np.arange(0.01, 0.99, 0.01)

ks_table = ks_by_cutoff(
    abt01_test[TARGET],
    abt01_test['PD'],
    cutoffs
)

ks_table.sort_values('KS', ascending=False).head(50)


### Matriz de confusão

In [ ]:
# define cutoff padrão (ajuste depois por negócio)
abt01_train['PRED'] = (abt01_train['PD'] >= cutoff).astype(int)
abt01_test['PRED']  = (abt01_test['PD'] >= cutoff).astype(int)

# calcula matriz de confusão
cm = confusion_matrix(abt01_test[TARGET], abt01_test['PRED'])

# plota matriz de confusão
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Bom (0)', 'Mau (1)']
)

plt.figure()
disp.plot()
plt.title('Confusion Matrix - Teste')
plt.show()

### Precision, Recall, F1

In [ ]:
# relatório de classificação
print(classification_report(
    abt01_test[TARGET],
    abt01_test['PRED'],
    digits=4
))

In [ ]:
# consolida métricas de performance (AUC, KS, Gini e Confusion Matrix) em um único dataset

# AUC
auc_train = roc_auc_score(abt01_train[TARGET], abt01_train['PD'])
auc_test  = roc_auc_score(abt01_test[TARGET],  abt01_test['PD'])

# KS
def calculate_ks(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return np.max(tpr - fpr)

ks_train = calculate_ks(abt01_train[TARGET], abt01_train['PD'])
ks_test  = calculate_ks(abt01_test[TARGET],  abt01_test['PD'])

# Gini
gini_train = 2 * auc_train - 1
gini_test  = 2 * auc_test  - 1

# Confusion Matrix (teste)
cm = confusion_matrix(abt01_test[TARGET], abt01_test['PRED'])
tn, fp, fn, tp = cm.ravel()

# Dataset final de comparação
metrics_df = pd.DataFrame([
    {
        'Base': 'Treino',
        'AUC': auc_train,
        'KS': ks_train,
        'Gini': gini_train,
        'Cutoff': cutoff,
        'TP': np.nan,
        'FP': np.nan,
        'TN': np.nan,
        'FN': np.nan,
        'Versao': VERSAO,
        'Data Execucao': DATA_EXECUCAO
    },
    {
        'Base': 'Teste',
        'AUC': auc_test,
        'KS': ks_test,
        'Gini': gini_test,
        'Cutoff': cutoff,
        'TP': tp,
        'FP': fp,
        'TN': tn,
        'FN': fn,
        'Versao': VERSAO,
        'Data Execucao': DATA_EXECUCAO
    }
])

In [ ]:
# salva métricas: atualiza se a versão existir, senão faz append
METRICS_PATH = METRICS_DIR / 'model_metrics_comparison.csv'

if METRICS_PATH.exists():
    # carrega histórico existente
    metrics_hist = pd.read_csv(METRICS_PATH)

    # remove registros da mesma versão (update)
    metrics_hist = metrics_hist[metrics_hist['Versao'] != metrics_df['Versao'].iloc[0]]

    # concatena com a versão atual
    metrics_final = pd.concat([metrics_hist, metrics_df], ignore_index=True)
else:
    # primeiro salvamento
    metrics_final = metrics_df.copy()

# salva resultado final
metrics_final.to_csv(METRICS_PATH, index=False)

metrics_final
print(f'✓ Métricas salvas/atualizadas em: {METRICS_PATH}')


In [ ]:
metrics_path = METRICS_DIR / 'model_metrics_comparison.csv'
metrics_df = pd.read_csv(metrics_path)

metrics_df